# CMIP6 core fix-plan example

This notebook uses a tiny synthetic CMIP6 dataset and a discovered fix plan to show the public Woodpecker plan API: inspect plan content, check a dataset with a plan, dry-run the plan, apply it in memory, and re-check the result.

The plan is bundled with Woodpecker and discovered by id, so user code does not need to know the plan file path.


In [ ]:
import numpy as np

import woodpecker
from woodpecker.testing import make_cmip6

Create a deterministic CMIP6-like dataset where near-surface air temperature is stored in Celsius instead of Kelvin.

In [ ]:
dataset = make_cmip6(overrides={"units": "degC"}, seed=7)
original_values = dataset["tas"].values.copy()

dataset

Load the discovered fix plan by id and inspect its content. The plan matches CMIP6-like datasets and selects the built-in core units fix.


In [ ]:
plan = woodpecker.plan.get("cmip6.core_units")

plan.model_dump()

In [ ]:
findings = woodpecker.plan.check(dataset, plan)

findings.fix_ids

A dry run reports what would change but leaves the dataset untouched.

In [ ]:
preview = woodpecker.plan.fix(dataset, plan, dry_run=True)

preview.stats, dataset["tas"].attrs["units"], np.allclose(dataset["tas"].values, original_values)

Apply the same plan in memory with `dry_run=False`.

In [ ]:
write = woodpecker.plan.fix(dataset, plan, dry_run=False)

(
    write.stats,
    dataset["tas"].attrs["units"],
    np.allclose(dataset["tas"].values, original_values + 273.15),
)

In [ ]:
recheck = woodpecker.plan.check(dataset, plan)

bool(recheck)